## 3. Maintenance is an explicit finite action

The Alpha maintenance example imposes a distinction, observes it
unmet, then admits a witness. A uniform rule later erases that
distinction under the active profile; the next *invoked* maintenance
step introduces another witness. Compare reports, states, and event
histories. Two matching report values need not mean identical states
or histories. The final maintenance checkpoint is inert when the
requirement is already met. This is a bounded host algorithm, not
a self-interpreter or continuous autonomous process.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from hashlib import sha256
from pathlib import Path


def find_repositories():
    candidates = [Path.cwd(), *Path.cwd().parents]
    extra = os.environ.get("GENESIS_REPOS_HOME")
    if extra:
        candidates.insert(0, Path(extra))
    for root in candidates:
        has_genesis = (root / "genesis" / "3_alpha_mathematical_language" / "Cargo.toml").is_file() or (root / "3_alpha_mathematical_language" / "Cargo.toml").is_file()
        has_symbol = (root / "the_symbol_game" / "education").is_dir() or (root / "education").is_dir()
        if has_genesis and has_symbol:
            g_root = root / "genesis" if (root / "genesis").is_dir() else root
            s_root = root / "the_symbol_game" if (root / "the_symbol_game").is_dir() else root
            return g_root, s_root
    raise RuntimeError(
        "Place genesis and the_symbol_game beside each other or set GENESIS_REPOS_HOME"
    )


GENESIS, SYMBOL = find_repositories()
REPOS_HOME = GENESIS.parent
ALPHA = Path(os.environ.get("GENESIS_ALPHA_ROOT", GENESIS / "3_alpha_mathematical_language"))
_PFP_PAPERS = (
    SYMBOL
    / "atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
)
PAPER33 = _PFP_PAPERS / "005_pfp_practitioners_field_manual" / "code"
PAPER32 = _PFP_PAPERS / "004_pfp_native_verification_and_evidence_custody" / "code"


def digest(path):
    return sha256(Path(path).read_bytes()).hexdigest()


source_id = {
    "alpha_front_end_sha256": digest(ALPHA / "src" / "lib.rs"),
    "paper032_governor_sha256": digest(PAPER32 / "l1_symbol_game_governor.py"),
    "paper033_toolkit_sha256": digest(PAPER33 / "symbol_game_toolkit.py"),
}
print(json.dumps(source_id, indent=2))


In [ ]:
cargo = shutil.which("cargo")
if cargo is None:
    raise RuntimeError("Rust/Cargo is required for this source-backed lab")
TARGET = Path(tempfile.gettempdir()) / "genesis_alpha_omega_practicum_target"
build_env = dict(os.environ, CARGO_TARGET_DIR=str(TARGET))
build = subprocess.run(
    [cargo, "build", "--offline", "--quiet", "--bin", "alpha"],
    cwd=ALPHA,
    env=build_env,
    capture_output=True,
    text=True,
    timeout=180,
)
if build.returncode:
    raise RuntimeError("Alpha build refused: " + build.stderr)
binary = TARGET / "debug" / ("alpha.exe" if os.name == "nt" else "alpha")


def run_alpha(source):
    with tempfile.TemporaryDirectory(prefix="alpha-practicum-") as tmp:
        path = Path(tmp) / "lesson.alpha"
        path.write_text(source, encoding="utf-8")
        result = subprocess.run(
            [str(binary), str(path)], cwd=ALPHA, capture_output=True, text=True, timeout=30
        )
    return {
        "source_sha256": sha256(source.encode("utf-8")).hexdigest(),
        "binary_sha256": digest(binary),
        "exit": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


first_source = (ALPHA / "examples" / "first.alpha").read_text(encoding="utf-8")
first = run_alpha(first_source)
assert first["exit"] == 0, first["stderr"]
assert "replay=checked" in first["stdout"]
print(first["stdout"])

In [ ]:
summary_pattern = re.compile(r"finished: data=(\d+) forms=(\d+) events=(\d+) replay=checked")


def counts(receipt):
    match = summary_pattern.search(receipt["stdout"])
    if receipt["exit"] != 0 or match is None:
        raise ValueError("no successful Alpha replay summary")
    return tuple(map(int, match.groups()))


learner_counts = (2, 1, 6)  # Edit after your prediction.
extra_check = run_alpha(first_source + "\ncheck data 2\n")
score_a = int(learner_counts == counts(first)) * 2 + int(counts(extra_check) == counts(first)) * 2
assert score_a == 4, (counts(first), counts(extra_check))
print("A:", score_a, "/4; baseline and extra-check counts:", counts(first), counts(extra_check))

In [ ]:
maintenance_source = (ALPHA / "examples" / "maintenance_cycle.alpha").read_text(encoding="utf-8")
maintenance = run_alpha(maintenance_source)
assert maintenance["exit"] == 0, maintenance["stderr"]
assert counts(maintenance) == (2, 2, 8)
maintain_lines = [
    line for line in maintenance["stdout"].splitlines() if "maintenance separated=" in line
]
assert sum("emitted=2" in line for line in maintain_lines) == 2
assert sum("emitted=0" in line for line in maintain_lines) == 1
assert "reports same=true" in maintenance["stdout"]
assert "states same=false" in maintenance["stdout"]
assert "histories same=false" in maintenance["stdout"]
assert "states same=true" in maintenance["stdout"]
assert "histories same=true" in maintenance["stdout"]
print("\n".join(maintain_lines))
print("terminal:", counts(maintenance))

**Assessment M (4 points).** Trace the eight events and distinguish
the two `emitted=2` repairs from the `emitted=0` checkpoint. Then test
a new presentation domain, cookies: two labels are supplied as data,
a requirement is imposed, and a witness is built. Nothing in the
interpreter validates claims about baking; the outside meaning of
these tokens is supplied by the lesson. Predict why deleting `impose`
must refuse maintenance.


In [ ]:
cookies = (
    "admit datum plain <plain_cookie>\n"
    "admit datum iced <iced_cookie>\n"
    "impose plain iced\n"
    "check collapsed plain iced\n"
    "maintain separation plain iced as sugar_witness\n"
    "check separate plain iced\n"
    "replay\n"
)
cookie_run = run_alpha(cookies)
assert cookie_run["exit"] == 0, cookie_run["stderr"]
assert counts(cookie_run) == (2, 1, 5)
no_requirement = run_alpha(cookies.replace("impose plain iced\n", ""))
assert no_requirement["exit"] == 1
assert (
    "maintenance" in no_requirement["stderr"].lower()
    or "requirement" in no_requirement["stderr"].lower()
)
print("cookie fixture:", counts(cookie_run))
print("without requirement:", no_requirement["stderr"].strip())
score_m = 4
print("M:", score_m, "/4")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `13_alpha_to_core_execution`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
